<a href="https://colab.research.google.com/github/Innovatewithapple/RAGPractice/blob/main/TaleStoryRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import random
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
   random.seed(seed)
   os.environ['PYTHONHASHSEED'] = str(seed)
   np.random.seed(seed)

   torch.manual_seed(seed)

   if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 💥 THE EXPERT TRICK: Force CUDA core algorithms to stay 100% rigid [prev]
    # This stops the GPU from switching algorithms to save time, ensuring absolute repeatability
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
setProductionSeed(isActive=True,seed=42)

In [2]:
!pip install -q pypdf

In [3]:
import pypdf

In [4]:
def extract_and_chunkpdf(pdfPath,chunksize,overlap):
  print(f'Opening pdf from path: {pdfPath}')

  reader = pypdf.PdfReader(stream=pdfPath)

  #unpack text sequentially page by page
  full_text_list = []

  for page in reader.pages[1:]:
    text = page.extract_text()
    if text: # System check: ignore if there is any image or graphics
      full_text_list.append(text)

  full_raw_string = " ".join(full_text_list)

  #Run the sliding window splitter loop
  words = full_raw_string.split()
  chunks = []

  for i in range(0,len(words),chunksize-overlap):
    chunk_words = words[i:i + chunksize]
    paragraph_string = " ".join(chunk_words)
    chunks.append(paragraph_string)

  return chunks

In [5]:
knowledge_strings = extract_and_chunkpdf(pdfPath='/content/Tell-Tale_Heart.pdf',chunksize=300,overlap=80)
print(f'total overlapped chunks: {len(knowledge_strings)}')
print("\n--- SAMPLE VIEW OF INDEX 0 PARAGRAPH ---")
print(knowledge_strings[0][:400] + "...") # Preview the first 400 characters

Opening pdf from path: /content/Tell-Tale_Heart.pdf
total overlapped chunks: 11

--- SAMPLE VIEW OF INDEX 0 PARAGRAPH ---
COPYRIGHT INFORMATION Short Story: “The Tell-Tale Heart” Author: Edgar Allan Poe, 1809–49 First published: 1843 The original short story is in the public domain in the United States and in most, if not all, other countries as well. Readers outside the United States s hould check their own countries’ copyright l aws to b e ce rtain they can legally download this e-story. The Online Books Page has a...


In [6]:
!pip install -q datasets transformers

In [7]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM
from tqdm import tqdm
import torch.nn.functional as F
from datasets import Dataset as HFDataset

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## **Encoder**

In [9]:
encoder_autoToken = AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
max_len = 512

In [11]:
encoder_token = encoder_autoToken(text=knowledge_strings,padding='max_length',max_length=max_len,add_special_tokens=True,return_attention_mask=True,truncation=True,return_tensors='pt')

In [12]:
class RAGKnowledgeDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask
    }

In [13]:
encoded_Data = RAGKnowledgeDataset(encoder_token)

In [14]:
encodedData_Loader = DataLoader(dataset=encoded_Data,batch_size=16,shuffle=False,pin_memory=True,num_workers=2)

In [15]:
class BERTWriter(nn.Module):
  def __init__(self):
    super().__init__()
    self.bert = AutoModel.from_pretrained('BAAI/bge-small-en-v1.5')

  def forward(self,input_ids,attention_mask):
    output = self.bert(input_ids=input_ids,attention_mask=attention_mask)
    x = output.last_hidden_state

    mean = attention_mask.unsqueeze(-1).float()
    mean = (x*mean).sum(dim=1) / mean.sum(dim=1)
    return mean

In [16]:
encoder_Model = BERTWriter().to(device)
encoder_Model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTWriter(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine

In [17]:
database_vectors = []

with torch.no_grad():
  with torch.amp.autocast('cuda'):

    for batch in tqdm(encodedData_Loader,desc=f'Encoding Batchs'):
      input_ids,attention_mask = batch['input_ids'].to(device) , batch['attention_mask'].to(device)

      encoder_model_output = encoder_Model(input_ids,attention_mask)

      database_vectors.append(encoder_model_output.cpu()) # move it to cpu so gpu rams are clean

vector_Database = torch.cat(database_vectors,dim=0)
print(f'vector database shape: {vector_Database.shape}')

/tmp/ipykernel_61348/3113097907.py:4: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):
Encoding Batchs:   0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Encoding Batchs: 100%|██████████| 1/1 [00:12<00:00, 12.96s/it]

vector database shape: torch.Size([11, 384])


In [18]:
questions = [
    "Who did the narrator kill?",
    "Why did he hate the old man?",
    "What made the narrator nervous?",
    "What happened to the body?",
    "Why did the narrator confess?",
    "What sound did he keep hearing?",
    "Where did he hide the body?"
]

In [19]:
#Tokenize the question
question_tokenize = encoder_autoToken(text=questions[1],padding='max_length',max_length=max_len,add_special_tokens=True,return_attention_mask=True,return_tensors='pt').to(device)

In [20]:
#Get input_ids and attention_mask from question token and put it in our encodedmodel
with torch.no_grad():
  with torch.amp.autocast('cuda'):
    query_vector = encoder_Model(question_tokenize['input_ids'],question_tokenize['attention_mask'])

query_vector_cpu = query_vector.cpu()

/tmp/ipykernel_61348/3954375916.py:3: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):


In [21]:
#Matrix multiplication where we multiply [1,768] with [768,20]
#Normalise our vector db and vector query
normalize_query = F.normalize(query_vector_cpu,p=2,dim=-1)
normalize_db = F.normalize(vector_Database,p=2,dim=-1)

similarity_scores = normalize_query @ normalize_db.T

best_score = torch.max(similarity_scores)
best_index = torch.argmax(similarity_scores)

print(best_score)

tensor(0.6507)


In [22]:
print(best_index)
print(f'Answer: {knowledge_strings[best_index]}')

tensor(1)
Answer: his gold I had no d esire. I think it was his eye! yes, it was this! One of his eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when I had made an opening sufficient for my head, I put in a dark lantern, all closed, closed, so that no light shone out, and then I thrust in my head. Oh, you would have laughed to see how cunningly I thrust it in! I moved it slowly—very, very slowly, s

In [23]:
print(knowledge_strings[1])

his gold I had no d esire. I think it was his eye! yes, it was this! One of his eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when I had made an opening sufficient for my head, I put in a dark lantern, all closed, closed, so that no light shone out, and then I thrust in my head. Oh, you would have laughed to see how cunningly I thrust it in! I moved it slowly—very, very slowly, so that I might not

# **Decoder**

In [24]:
decoder_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')

In [25]:
decoder_Model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct',device_map='auto')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [26]:
decoder_Model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [27]:
messages = [
    {
        "role": "system",
        "content": f"""You must answer ONLY using exact information from the context.

Do not interpret.
Do not explain symbolism.
Do not infer emotions unless explicitly stated.

If information is not directly stated, say:
"Not explicitly stated in the context."\n\nContext:\n{knowledge_strings[best_index]}"""
    },
    {
        "role": "user",
        "content": questions[1]
    }
]

text_promt = decoder_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
print(text_promt)
input_ids_decoder = decoder_tokenizer.encode(text_promt,return_tensors='pt').to(device)
attention_mask = torch.ones_like(input_ids_decoder).to(device)

<|im_start|>system
You must answer ONLY using exact information from the context.

Do not interpret.
Do not explain symbolism.
Do not infer emotions unless explicitly stated.

If information is not directly stated, say:
"Not explicitly stated in the context."

Context:
his gold I had no d esire. I think it was his eye! yes, it was this! One of his eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when

In [28]:
with torch.no_grad():
  with torch.amp.autocast('cuda'):
    output_ids = decoder_Model.generate(
        input_ids_decoder,
        attention_mask=attention_mask,
        max_new_tokens=60,       # Gives it plenty of room to write a complete sentence [prev]
        do_sample=True,          # Enables natural conversational sampling [prev]
        top_k=10,                # Restricts the word pool to keep it highly focused [prev]
        temperature=0.1,         # Drops creativity near zero to lock onto context facts [prev]
        repetition_penalty=1.15, # Nudges it away from loops without breaking grammar rules [prev]
        no_repeat_ngram_size=0,  # Turned off: Disables the rigid, destructive phrase bans completely [prev]
        pad_token_id=decoder_tokenizer.eos_token_id
    )

    newTokens = output_ids[0,len(input_ids_decoder):]
    final_output = decoder_tokenizer.decode(newTokens,skip_special_token=True)

/tmp/ipykernel_61348/2658285959.py:2: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):


In [29]:
print(final_output)

system
You must answer ONLY using exact information from the context.

Do not interpret.
Do not explain symbolism.
Do not infer emotions unless explicitly stated.

If information is not directly stated, say:
"Not explicitly stated in the context."

Context:
his gold I had no d esire. I think it was his eye! yes, it was this! One of his eyes resembled that of a vulture—a pale blue eye, with a film over it. Whenever it fell upon me, my blood ran cold; and so by degrees—very gradually—I made up my mind to take the life of the old man, and thus rid myself of the eye for ever. Now this is the point. You fancy me mad. Madmen know nothing. But you should h ave seen me. You should have seen h ow w isely I proceeded—with what caution— with what foresight—with what dissimulation I went to work! I was never kinder to the old man than du ring the whole week b efore I killed h im. And every night, about midnight, I turned the latch of his door and opened it—oh, so gently! And then, when I had made 